# Learning Objectives

* Specify what compute backend you want to use for these training notebooks.


# Introduction

Pegasus allows you to define a workflow and run the same workflow against different compute resources. By default, these training notebooks are setup to run the jobs in your workflows on the TestPool that is part of the ACCESS Pegasus setup (https://pegasus.access-ci.org/). 

However, you can choose to run these notebooks against different supported compute backends from a workflow submit host where Pegasus and HTCondor are deployed. This is done by telling Pegasus what `Site Catalog` to use. While the concept of the `Site Catalog` is covered in `04-Tutorial-Site-Layout`, if you are doing this training NOT on ACCESS Pegasus, then you need to specify the `Site Catalog` to use from amongst the one specified in this [GitHub repo](https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf).

The resource descriptions are continuously being added. Some of the popular ones in the repo are 

* `access-pegasus.yml`: ACCESS Pegasus Test Pool for doing training. 
* `access-pegasus-expanse.yml`: SDSC Expanse from ACCESS Pegasus.
* `nersc-perlmutter.yml`: NERSC Perlmutter from any workflow submit host.
* `ospool-ap40.yml`: OS Pool from ACCESS Point AP40.
* `usc-discovery.yml`: USC Discovery HPC Cluster.
* `usc-laguna.yml`: SoCal Research Computing Alliance Laguna Cluster.
* `unity.yml`: MGHPCC Unity HPC Platform. 

# 1. Define the Resource/Compute Backend to Use

In this step, we setup some resource backend specific information in the default Pegasus configuration file in $HOME/.pegasusrc

In [ ]:
import os
import sys
import logging
import argparse
from pathlib import Path

from Pegasus.api import *

logging.basicConfig(level=logging.DEBUG)

# download the site catalog file to use from
# https://github.com/pegasushub/pegasus-site-catalogs/tree/main/conf     
# Update it to any one specified at the above URL.
RESOURCE_SITE_CATALOG = "access-pegasus.yml"

# if you are jobs are going to run on a HPC
# resource such as an ACCESS Cluster, or NERSC Perlmutter
# Specify, your username and project on that 
# resource.
RESOURCE_USERNAME = ""
RESOURCE_PROJECT =  ""

# write out the properties, so that it can be picked up
# when you plan the workflow as part of the subsequent
# training notebooks.
props = Properties()
props["pegasus.catalog.site.repo.file"] = RESOURCE_SITE_CATALOG
props["env.RESOURCE_USERNAME"] = RESOURCE_USERNAME
props["env.RESOURCE_PROJECT"] = RESOURCE_PROJECT
props.write(Path.home() / ".pegasusrc")

Now lets inspect the generated pegasus configuration file. It will contain the properties listed as simple key value pairs.

In [ ]:
!cat ~/.pegasusrc

# 2. Additional Setup for Resource/Compute Backends

## 2.1 NERSC Perlmutter Setup

We need to define some additional configuration for running workflows on NERSC such as 

* SFAPI tokens for job submission
* SSH key for data transfers
* Specifying high performant scratch filesystem

### 2.1.1 Superfacility API token generation

Sometimes, the resource that you need to submit workflows to may require you to generate credentials/tokens to submit to the resource. One of such compute backends is the `Perlmutter` cluster at NERSC. To be able to submit jobs to it, you need to generate a sfapi client token in the `NERSC IRIS` portal. 

To do that, please follow the instructions available in the [Pegasus Documentation](https://pegasus.isi.edu/docs/5.1.3-dev.0/user-guide/deployment-scenarios.html#nersc-perlmutter-via-sfapi).

<div class="alert alert-block alert-info">
<b>Note:</b> When creating the SFAPI token, please make sure the IP address that you whitelist in IRIS is the workflow submit node i.e. the host where you are running these notebooks.
</div>

### 2.1.2 Generation of SSH key for data transfers

NERSC has developed a service, called [sshproxy](https://docs.nersc.gov/connect/mfa/#sshproxy), that allows you to use MFA to get an SSH key that is valid for a limited time (24 hours by default). sshproxy provides a type of single-sign-on capability for SSH to NERSC systems.
We use that to generate a ssh key that is used for data transfers to Perlmutter. 

To do this open a terminal by clicking New -> Terminal in the Jupyter menu from where you launched this notebook. 

Once in the shell/terminal execute the following command

```
sshproxy -u RESOURCE_USERNAME
```

Substitute RESOURCE_USERNAME with your username at NERSC. It will ask you for the NERSC password+OTP

Once done you will have a key named nersc in your ~/.ssh directory.
Lets check that by checking the validity of your nersc ssh key.

In [ ]:
!ssh-keygen -L -f ~/.ssh/nersc-cert.pub | grep Valid

### 2.1.3 Specify high performant scratch filesystem

NERSC recommends using the scratch filesystem to run your jobs. In order to do this, you need to specify the variable `RESOURCE_SCRATCH_DIR` in your properties file. This value gets used when reading in the [NERSC resource description](https://github.com/pegasushub/pegasus-site-catalogs/blob/main/conf/nersc-perlmutter.yml).

In [ ]:
if RESOURCE_SITE_CATALOG  == "nersc-perlmutter.yml":
    RESOURCE_SCRATCH_DIR = "/pscratch/sd/{}/{}".format(RESOURCE_USERNAME[0], RESOURCE_USERNAME) 
    print( "RESOURCE_SCRATCH_DIR is {}".format(RESOURCE_SCRATCH_DIR))
    # write out the properties, so that it can be picked up
    # when you plan the workflow as part of the subsequent
    # training notebooks.
    props = Properties.load(Path.home() / ".pegasusrc")
    props["env.RESOURCE_SCRATCH_DIR"] = RESOURCE_SCRATCH_DIR
    props.add_site_profile("local", "pegasus", "SSH_PRIVATE_KEY", Path.home()/ ".ssh/nersc")
    props.write(Path.home() / ".pegasusrc")

Now lets inspect the generated pegasus configuration file. It will contain the properties listed as simple key value pairs.

In [ ]:
!cat ~/.pegasusrc

## What's Next?

To continue exploring Pegasus, the next tutorial notebook  will show you how to construct a simple `hello->world` pipeline workflow using the Pegasus Workflow API.